# PlotlyBrain — workflow 

### QUINT output → Scores → Recolored Allen maps → Saving figures

In [ ]:
import os
import pandas as pd

from plotlybrain.scores import save_scores
from plotlybrain.coord_system import (
    ap_mm_to_slice_index, 
    slice_index_to_ap_mm,
)

from plotlybrain.plotly_render import (
    load_score,
    render_brain_slice, 
    export_brain_slice, 
)


## Compute a score table from QUINT exports

PlotlyBrain can summarize QUINT detections into different score tables:

- *frequency*, fraction of animals with objects > 0
- *rel_abundance* z-scored regional abundance
- *density* objects per unit area

This scores are combined with a metadata file that defines the experimental groups of our dataset.

In [ ]:
BASE_DIR = os.getcwd()  
DATA_DIR = os.path.join(BASE_DIR, "ad_dataset")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
META_PATH = os.path.join(DATA_DIR, "metadata.csv")

os.makedirs(OUT_DIR, exist_ok=True)

Getting the score table

In [ ]:
SCORE = "rel_abundance"
df_scores = save_scores(
    data_dir=DATA_DIR,
    metadata_path=META_PATH,
    out_path=OUT_DIR,
    metadata_sep=";",
    group_col='group',
    score=SCORE,
    rel_abundance_method="reference",
    reference_mode="pooled",
)

## Load score lookups 

Once the score tables have been written to disk, we can load one grouped score file and convert it into lookpup dictionaries, with the following function: 

In [ ]:
VALUE_COL = "relative_abundance_z" # --> EDIT
GROUP = "3m" # --> EDIT

data_path = os.path.join(OUT_DIR, f"rel_abundance_{GROUP}.csv")
id2value, id2name, score_col = load_score(
    data_path,
    id_col="Region ID",
    name_col="Region name",
    value_col=VALUE_COL,
)

## Converting AP Bregma Level to Allen slice Index

The Allen atlas is organized by slice indices, while experimental data are typically referenced using AP coordinates relative to Bregma. This step converts a selected AP value (in mm) into the corresponding coronal slice index at the chosen atlas resolution.

In [ ]:
ap_mm = -2.0
slice_index = ap_mm_to_slice_index(ap_mm, resolution_um=25)
print("slice_index:", slice_index)


## Build a GeoJSON slice from the Allen annotation volume

Loading the Allen annotation volume and the ontology file, extract one coronal slice and convert its labeled regions into GeoJSON polygons. This representation is what PlotlyBrain uses for rendering. 

In [ ]:
OUT_DIR = "/Users/annateruel/Desktop/allen_cache"

volume, header = load_annotation_volume(
    resolution_um=25,
    storage_mode="disk",
    cache_dir=OUT_DIR,
    overwrite=False,
)

structure_df = load_structure_graph(
    storage_mode="disk",
    cache_dir=OUT_DIR,
    overwrite=False,
)

slice_img = get_slice_view(volume, slice_index, "coronal")
gj = build_slice_geojson(
    slice_img=slice_img,
    structure_df=structure_df,
    slice_index=slice_index,
    orientation="coronal",
    resolution_um=25,
    min_area_px=5,
    simplify_px=0.5, 
    polygon_mode="contour", 
    smooth_sigma=1.0, 
)

## Plotly GeoJSON render
Render a single coronal slice using Plotly.


In [ ]:
fig = render_brain_slice(
    geojson_obj=gj,
    id2value=id2value,
    id2name=id2name,
    title=f"Brain slice | AP={ap_mm} mm | idx={slice_index}",
    score_name=score_col,
    line_color="white",
    line_width=0.8,
    line_shape="spline",
    smoothing=0.7,
    show_colorbar=True,
)
fig.show()

Exporting the slice image to SVG

In [ ]:
out_path = os.path.join(OUT_DIR,f"slice_{slice_index}_{SCORE}.svg")
export_brain_slice(
    fig,
    out_path=out_path,
    width=1200,
    height=1200,
    scale=2,
)

## Ploting multiple Bregma levels

Render multiple coronal slice using Plotly, based on a list of AP Bregma levels. 


In [ ]:
ap_levels = [+3.3, +0.98, -1.94, -2.92, -3.80]
for ap_mm in ap_levels:
    slice_index = ap_mm_to_slice_index(
        ap_mm=ap_mm,
        resolution_um=25,
    )

    slice_img = get_slice_view(
        volume,
        slice_index,
        "coronal",
    )

    gj = build_slice_geojson(
        slice_img=slice_img,
        structure_df=structure_df,
        slice_index=slice_index,
        orientation="coronal",
        resolution_um=25,
        min_area_px=5,
        simplify_px=0.8,
    )

    fig = render_brain_slice(
        geojson_obj=gj,
        id2value=id2value,
        id2name=id2name,
        # colorscale = 'Viridis',
        title=f"AP {ap_mm:.2f} mm",
        score_name=score_col,
        line_color="white",
        line_width=0.8,
        line_shape="spline",
        smoothing=0.7,
        show_colorbar=True,
    )
    ap_label = f"{ap_mm:+.2f}".replace(".", "p")  # -2.00 → -2p00
    filename = f"slice_{slice_index}_ap_{ap_label}_{score_col}.svg"

    out_path = os.path.join(OUT_DIR, filename)
    export_brain_slice(
        fig,
        out_path=out_path,
        width=1200,
        height=1200,
        scale=2,
    )

    print(f"Saved: {out_path}")
    fig.show()